# Embedding Generation

This notebook generates EO embeddings from the Sentinel-2 forest disturbance patches using TerraTorch's `EmbeddingGenerationTask`.

Workflow:

1. load raw Sentinel-2 frames with `ForestDisturbanceDataModule`
2. run a pretrained TerraTorch FM backbone in Lightning `predict` mode
3. save embeddings as zarr timeseries

In [ ]:
import os
from pathlib import Path
import warnings

import lightning.pytorch as pl
import s2tutorial as s2t

warnings.filterwarnings("ignore")
pl.seed_everything(0)

In [ ]:
ROOT = Path(os.environ.get("S2T_DATA_ROOT", ".."))
OUTPUT_ROOT = Path(
    os.environ.get(
        "S2T_EMBEDDING_OUTPUT",
        ROOT / "embeddings_center_patch" / "terramind_v1_small"
    )
)

required = [
    ROOT / "samples.parquet",
    ROOT / "labels.parquet",
    ROOT / "frames.parquet",
    ROOT / "classes.json",
    ROOT / "patches",
]
missing = [path for path in required if not path.exists()]
if missing:
    raise RuntimeError("Missing required dataset paths:\n" + "\n".join(map(str, missing)))

print(f"Data root:   {ROOT}")
print(f"Output root: {OUTPUT_ROOT}")

In [9]:
class CenterCropTensor:
    """
    Center crop that aligns the image center
    to the center of a patch token.
    """

    def __init__(
        self,
        size: int | tuple[int, int],
        patch_size: int = 16,
    ) -> None:
        if isinstance(size, int):
            size = (size, size)

        self.height, self.width = size
        self.patch_size = patch_size

    def _compute_start(
        self,
        full_size: int,
        crop_size: int,
    ) -> int:
        center = full_size // 2

        center_start = (full_size - crop_size) // 2
        desired_mod = self.patch_size // 2
        valid_starts = range(0, full_size - crop_size + 1)

        candidate_starts = [
            s
            for s in valid_starts
            if (center - s) % self.patch_size == desired_mod
        ]

        if not candidate_starts:
            raise ValueError(
                f"Could not find valid crop alignment for "
                f"full_size={full_size}, crop_size={crop_size}, "
                f"patch_size={self.patch_size}"
            )

        return min(candidate_starts, key=lambda s: abs(s - center_start))

    def __call__(self, sample):
        image = sample["image"]

        _, h, w = image.shape

        top = self._compute_start(h, self.height)
        left = self._compute_start(w, self.width)

        sample["image"] = image[
            :,
            top : top + self.height,
            left : left + self.width,
        ]

        return sample

In [10]:
datamodule = s2t.ForestDisturbanceDataModule(
    root=ROOT,
    batch_size=8,
    num_workers=0,
    sample_mode="frames",
    label_mode="none",
    predict_split="all",
    sensor="s2_all",
    transforms=CenterCropTensor(224),
)

datamodule.setup("predict")
print(f"Frames to embed: {len(datamodule.predict_dataset)}")

batch = next(iter(datamodule.predict_dataloader()))
print("image:", batch["image"].shape)
print("filename:", batch["filename"][:3])

Frames to embed: 17357
image: torch.Size([8, 12, 224, 224])
filename: ['0_2016-11-10', '0_2016-12-20', '0_2017-01-12']


In [ ]:
from terratorch.tasks import EmbeddingGenerationTask

model = EmbeddingGenerationTask(
    model_args={
        "backbone": "terramind_v1_small",
        "backbone_modalities": ["S2L2A"],
        "backbone_pretrained": True,
    },
    output_format="zarr_series",
    output_dir=OUTPUT_ROOT,
    layers=[-1],
    embedding_pooling="annotated_patch",
)

In [12]:
import torch

images = batch["image"]

with torch.no_grad():
    outputs = model(images)

if not isinstance(outputs, list):
    outputs = [outputs]

print("input:", images.shape)
for i, out in enumerate(outputs):
    print(f"layer {i}:", out.shape)

input: torch.Size([8, 12, 224, 224])
layer 0: torch.Size([8, 196, 384])


In [ ]:
trainer = pl.Trainer(accelerator="auto")
_ = trainer.predict(model, datamodule=datamodule)